# HGB Normal + Combined 최적화

**조장님 GitHub의 데이터 로드 코드와 전처리 함수를 그대로 불러와서** 모델만 HGB로 바꾼 노트북입니다. 그래서 분할·손글씨·CoM·크기/획 증강이 MLP 실험과 완전히 같아요.

**사용법: 고칠 곳 없이 위에서부터 순서대로 실행만 하면 됩니다.** 결과는 내 구글 드라이브 `HGB_results/` 폴더에 저장돼요.

| 셀 | 내용 | 보고서 절 |
|---|---|---|
| N1 | 원본 HGB (기준) | 기준 모델 |
| N2 | CoM 정렬 + 교차평가 | 중심 정렬 |
| N3 | CoM + 크기 30%·획 28% 증강 | 균형 증강 |
| N4 | 하이퍼파라미터 3개를 하나씩 | 하이퍼파라미터 |
| C1 | Combined 작성자 보류 4-fold OOF | Combined |

## 0. 저장소 가져오기 + 드라이브 연결

In [ ]:
!test -d /content/Machine_Learning || git clone --depth 1 https://github.com/ryu980920/Machine_Learning.git /content/Machine_Learning

from google.colab import drive
drive.mount('/content/drive')
import os; OUT = '/content/drive/MyDrive/HGB_results/'; os.makedirs(OUT, exist_ok=True)

import sys, time, numpy as np, pandas as pd
from pathlib import Path
REPO = Path('/content/Machine_Learning')
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Combined/scripts'))
sys.path.insert(0, str(REPO/'05_model_optimize/MLP/Normal/scripts'))

# 조장님 코드 그대로 사용
from combined_mlp_writer_cv import load_official, load_writers
from exp006_size_com_comparison import process          # CoM 정렬
import exp_cmb005_balanced_scale_stroke as L             # prepareraw: 크기·획 증강 + CoM

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

## 1. 데이터 로드 (공식 Train 76,800 / Validation 19,200, 손글씨 4명 × 640장)

In [ ]:
off = load_official(REPO/'02_data/datasets/MNIST_EMNIST_official_split_112000.npz')
wr  = load_writers(REPO/'02_data/Hand_made')
WRITERS = list(L.WRITERS); LABELS = '0123456789ABCDEF'
ytr, yva = off['y_train'], off['y_val']
yhm = np.concatenate([wr[w]['y'] for w in WRITERS])
whm = np.concatenate([[w]*len(wr[w]['y']) for w in WRITERS])
print(off['X_train'].shape, off['X_val'].shape, len(yhm))   # (76800,784) (19200,784) 2560

## 2. 공통 함수 (평가·학습)

In [ ]:
PAIRS = [('D','0'), ('0','D'), ('B','8'), ('8','B'), ('5','F'), ('7','9')]
results = []

def hm_scores(pred, ytrue, writers):
    isd = ytrue < 10
    r = dict(hm_acc=accuracy_score(ytrue, pred), hm_f1=f1_score(ytrue, pred, average='macro'),
             hm_digit=accuracy_score(ytrue[isd], pred[isd]), hm_letter=accuracy_score(ytrue[~isd], pred[~isd]))
    for w in WRITERS: r[f'w{w}'] = accuracy_score(ytrue[writers==w], pred[writers==w])
    for a, b in PAIRS: r[f'{a}->{b}'] = int(np.sum((ytrue==LABELS.index(a)) & (pred==LABELS.index(b))))
    return r

def evaluate(name, model, Xva, Xhm, extra=None):
    pv = model.predict(Xva); ph = model.predict(Xhm)
    row = dict(name=name, val_acc=accuracy_score(yva, pv), val_f1=f1_score(yva, pv, average='macro'),
               **hm_scores(ph, yhm, whm), **(extra or {}))
    results.append(row); pd.DataFrame(results).to_csv(OUT+'HGB_results.csv', index=False)
    print({k:(round(v,4) if isinstance(v,float) else v) for k,v in row.items()})
    return ph

def hgb(**kw):
    p = dict(max_iter=300, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, random_state=42)
    p.update(kw); return HistGradientBoostingClassifier(**p)

def fit(m, X, y):
    t = time.time(); m.fit(X, y); return dict(fit_s=round(time.time()-t), iters=m.n_iter_)

## N1. 원본 (전처리 없음, 기준 모델)

In [ ]:
Xhm_raw = np.vstack([wr[w]['X'] for w in WRITERS])
m1 = hgb(); info = fit(m1, off['X_train'], ytr)
ph1 = evaluate('N1_raw', m1, off['X_val'], Xhm_raw, info)

## N2. CoM 정렬 (학습·검증·손글씨 모두 같은 처리)
+ 교차평가: 원본 모델에 CoM 입력만 넣기 → '이동 효과'와 '재학습 효과' 분리

In [ ]:
Xtr_com, tw_com, _, _ = L.prepareraw(off, wr, 'com_only')
Xva_com = process(off['X_val'], 'com', 20)
Xhm_com = np.vstack([tw_com[w] for w in WRITERS])

m2 = hgb(); info = fit(m2, Xtr_com, ytr)
evaluate('N2_CoM', m2, Xva_com, Xhm_com, info)
evaluate('N1모델 + CoM입력 (교차)', m1, Xva_com, Xhm_com)

## N3. CoM + 크기 30%·획 28% 증강 (조장님 Combined 최종 전처리)

In [ ]:
Xtr_aug, tw_aug, info_aug, _ = L.prepareraw(off, wr, 'scale30_stroke28')
print(info_aug)
m3 = hgb(); info = fit(m3, Xtr_aug, ytr)
evaluate('N3_CoM+size30_stroke28', m3, Xva_com, Xhm_com, info)
pd.DataFrame(results)

## N4. 하이퍼파라미터 — 하나씩 바꾸고, 공식 Validation이 가장 높은 값을 고정한 뒤 다음으로
아래 `USE_AUG`는 N2/N3 중 손글씨 정확도가 더 좋았던 쪽으로 두세요.

In [ ]:
USE_AUG = True                    # N3가 N2보다 좋으면 True, 아니면 False
Xtr_best = Xtr_aug if USE_AUG else Xtr_com

best = {}
for pname, values in [('learning_rate', [0.05, 0.1, 0.2]),
                      ('max_leaf_nodes', [15, 31, 63]),
                      ('l2_regularization', [0.0, 1.0, 10.0])]:
    sc = {}
    for v in values:
        kw = {**best, pname: v}
        m = hgb(**kw); info = fit(m, Xtr_best, ytr)
        evaluate(f'N4_{pname}={v}', m, Xva_com, Xhm_com, {**info, **kw})
        sc[v] = results[-1]['val_acc']
    best[pname] = max(sc, key=sc.get); print('>>> 선택', pname, '=', best[pname])
print('Normal 최종 설정:', best)

## C1. Combined — 작성자 1명 보류 · 나머지 3명 1,920장을 학습에 추가 · 4번 반복 → OOF

In [ ]:
def combined_oof(Xtr, hm_map, kw, name):
    oof = np.zeros(len(yhm), dtype=int)
    for w in WRITERS:
        others = [o for o in WRITERS if o != w]
        X = np.vstack([Xtr] + [hm_map[o] for o in others])
        y = np.concatenate([ytr] + [wr[o]['y'] for o in others])
        m = hgb(**kw); fit(m, X, y)
        oof[whm == w] = m.predict(hm_map[w])
        print(f'  보류 {w}: {accuracy_score(yhm[whm==w], oof[whm==w]):.4f}')
    row = dict(name=name, **hm_scores(oof, yhm, whm), **kw)
    results.append(row); pd.DataFrame(results).to_csv(OUT+'HGB_results.csv', index=False)
    print(row); return oof

raw_map = {w: wr[w]['X'] for w in WRITERS}
oof_raw  = combined_oof(off['X_train'], raw_map, {}, 'C1_raw (전처리·튜닝 없음)')
oof_best = combined_oof(Xtr_best, tw_com, best, 'C1_best (Normal 최종 전처리·설정)')

## 혼동행렬 그림 + 최종 표

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(2, 2, figsize=(16, 15))
for a, (p, t) in zip(ax.ravel(), [(ph1,'N1 raw (Normal)'), (m3.predict(Xhm_com),'N3 aug (Normal)'),
                                  (oof_raw,'C1 raw (Combined OOF)'), (oof_best,'C1 best (Combined OOF)')]):
    ConfusionMatrixDisplay.from_predictions(yhm, p, labels=range(16), display_labels=list(LABELS),
                                            normalize='true', values_format='.2f', ax=a, colorbar=False)
    a.set_title(t)
plt.tight_layout(); plt.savefig(OUT+'HGB_confusion.png', dpi=130); plt.show()
pd.DataFrame(results)